In [22]:
import json
from kafka import KafkaProducer
import pandas as pd
import csv, gzip

from time import time

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

In [23]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

producer.bootstrap_connected()

True

In [24]:
def main():

    t0 = time()

    # Create a Kafka producer
    producer = KafkaProducer(
        bootstrap_servers='localhost:9092',
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )

    csv_file = 'data/green_tripdata_2019-10.csv.gz'  # change to your CSV file path if needed

    with gzip.open(csv_file, 'rt', newline='') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            message = {
                'lpep_pickup_datetime' : row[1],
                'lpep_dropoff_datetime': row[2],
                'PULocationID' : row[5],
                'DOLocationID' : row[6],
                'passenger_count' : row[7],
                'trip_distance' : row[8],
                'tip_amount' : row[12]
            }

            producer.send('green-trips', value=message)

    # Make sure any remaining messages are delivered
    producer.flush()
    producer.close()

    t1 = time()
    took = t1 - t0
    print(took)


if __name__ == "__main__":
    main()

22.015620231628418
